# Paso 8 — Evaluación

Notebook de verificación de los tres mecanismos de evaluación del motor (decisión A8, AGENTS.md):

1. **Golden set interno**: casos con resultado esperado, revisados a mano sobre productos reales
   (`evaluacion/golden_set.py`).
2. **Parity-check con Nutri-Score**: correlación entre el orden de D1 y Nutri-Score
   (`evaluacion/parity_check.py`). Divergencias grandes son señal de error, no de mérito.
3. **Diagnóstico de cobertura**: cuántos productos caen en la banda "información insuficiente" y
   por qué dimensión falta (`evaluacion/coverage_diagnostic.py`).

Mismo estilo que `03_transformacion_score.ipynb` y `04_modelo_recomendacion.ipynb`: cada módulo se
verifica aquí contra los datos reales del snapshot `off_csv_20260919`. Las cifras que aparecen en
las celdas de markdown se transcriben de la salida real de las celdas de código inmediatamente
anteriores, después de ejecutar el notebook — no son estimaciones.

Reutiliza los mismos 3 perfiles de ejemplo (Ana, Beto, Caro) que `04_modelo_recomendacion.ipynb`
para la Sección 4 (diagnóstico de cobertura), por continuidad narrativa.

In [1]:
from pathlib import Path

import duckdb
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from evaluacion.coverage_diagnostic import diagnosticar_cobertura
from evaluacion.golden_set import evaluar as evaluar_golden_set
from evaluacion.parity_check import UMBRAL_CORRELACION_RAZONABLE, calcular_parity_check
from nutrimatch.engine.constants import CORE8_NUTRIENTES
from nutrimatch.engine.nutrition_score import calcular_subpuntaje_d1
from nutrimatch.engine.preference_score import calcular_d3
from nutrimatch.engine.product_naming import resolver_nombre_producto
from nutrimatch.engine.user_weights import convertir_prioridades_a_pesos

SNAPSHOT_ID = "off_csv_20260919"
RUTA_PROCESADOS = Path("../datos/procesados")
RUTA_SNAPSHOT_CRUDO = Path("../datos/procesados/off_mexico_20260919.parquet")

con = duckdb.connect()

ruta_matriz = RUTA_PROCESADOS / "matriz_nut_100g_20260919.parquet"
matriz = con.execute(f"SELECT * FROM '{ruta_matriz}'").df()

# Igual que en 04_modelo_recomendacion.ipynb: columnas extra del snapshot crudo, más
# nutriscore_score/nutriscore_grade (necesarios aquí para el parity-check, Sección 3).
extra = con.execute(
    f"""
    SELECT code, product_name, generic_name, abbreviated_product_name,
           labels_tags, allergens, traces, ingredients_analysis_tags,
           nutriscore_score, nutriscore_grade
    FROM '{RUTA_SNAPSHOT_CRUDO}'
    """
).df()

df = matriz.merge(extra, on="code", how="left", validate="one_to_one")
df = df.rename(columns={"product_name": "product_name_bruto"})

# Decisión A28: fallback de nombre para mostrar (no afecta a ningún cálculo del motor).
nombres_resueltos = df.apply(
    lambda r: resolver_nombre_producto(r["product_name_bruto"], r["generic_name"], r["abbreviated_product_name"]),
    axis=1,
)
df["product_name"] = nombres_resueltos.apply(lambda t: t[0])

# Hallazgo (documentado como B14 en AGENTS.md): `nutriscore_score` llega como VARCHAR en el
# snapshot crudo (scripts/ingesta_off.py usa `all_varchar=true` a propósito: "la conversión de
# tipos se hace después, de forma auditada"). No es parte de CORE8, así que el paso 6 nunca lo
# tipó. Sin esta conversión explícita, cualquier comparación numérica (incluida la correlación de
# la Sección 2) se haría por orden LEXICOGRÁFICO ("9" > "51") en vez de numérico, en silencio.
valores_antes = df["nutriscore_score"].notna().sum()
df["nutriscore_score"] = pd.to_numeric(df["nutriscore_score"], errors="coerce")
valores_no_parseables = valores_antes - df["nutriscore_score"].notna().sum()
print(f"nutriscore_score: {valores_antes} valores no nulos antes de convertir a numérico, "
      f"{valores_no_parseables} no se pudieron parsear (deberían ser 0: son enteros limpios en OFF).")
assert valores_no_parseables == 0, "nutriscore_score trae valores no numéricos inesperados: revisar"

# D1 es agnóstico del usuario (Sección 4 de 04_modelo_recomendacion.ipynb): se calcula una sola vez.
d1_resultados = df.apply(
    lambda r: calcular_subpuntaje_d1({n: r[f"percentil_{n}"] for n in CORE8_NUTRIENTES}),
    axis=1,
)
df["d1"] = d1_resultados.apply(lambda t: t[0])

print("Filas tras el merge:", len(df))
print(f"D1 disponible: {df['d1'].notna().sum()} de {len(df)} productos ({df['d1'].notna().mean() * 100:.1f}%)")
print(f"D2 disponible: {df['d2'].notna().sum()} de {len(df)} productos ({df['d2'].notna().mean() * 100:.1f}%)")
df[["code", "product_name", "categoria_referencia", "d1", "d2", "nutriscore_grade", "nutriscore_score"]].head(3)

nutriscore_score: 6431 valores no nulos antes de convertir a numérico, 0 no se pudieron parsear (deberían ser 0: son enteros limpios en OFF).
Filas tras el merge: 16851
D1 disponible: 7040 de 16851 productos (41.8%)
D2 disponible: 6781 de 16851 productos (40.2%)


,code,product_name,categoria_referencia,d1,d2,nutriscore_grade,nutriscore_score
0,00000140,tisane saveur lunaire,en:herbal-teas,NaN,25.000000,unknown,NaN
1,00000154,Mct Oil Powder,NaN,NaN,22.222222,d,14.0
2,00000285,NaN,NaN,NaN,NaN,unknown,NaN


## Sección 1 — Golden set interno (decisión A8/A29)

13 casos sobre productos reales del snapshot, elegidos a mano para cubrir rutas concretas del
motor: D1/D2 en los dos extremos, los tres estados del filtro de alergia (confirmado en
`allergens`, "puede contener" en `traces`, sin dato), los tres estados de dieta vegana
(compatible, incompatible, "tal vez" → no verificable), independencia entre D1 y D2 cuando falta
solo una de las dos fuentes (categoría vs. NOVA), el fallback de nombre (A28), D3 con una etiqueta
exacta, y un caso de regresión de score completo ya auditado a mano en
`04_modelo_recomendacion.ipynb` (Sección 8). El detalle de cada caso y su justificación están en
`evaluacion/golden_set.py`; las mismas pruebas corren en `tests/test_golden_set.py`.

In [2]:
reporte_golden_set = evaluar_golden_set(df)

n_pasan = int(reporte_golden_set["paso"].sum())
print(f"Golden set: {n_pasan} de {len(reporte_golden_set)} casos pasan.\n")

pd.set_option("display.max_colwidth", 100)
display(reporte_golden_set[["code", "paso", "descripcion"]])

fallidos = reporte_golden_set[~reporte_golden_set["paso"]]
if not fallidos.empty:
    print("\nCasos fallidos (detalle):")
    for fila in fallidos.itertuples():
        print(f"- {fila.code}: {fila.fallos}")
else:
    print("\nTodos los casos pasan.")

assert n_pasan == len(reporte_golden_set), "el golden set no debe fallar en el snapshot de referencia"

Golden set: 13 de 13 casos pasan.



,code,paso,descripcion
0,7501000112425,True,"Donitas espolvoreadas: azúcar/sal/grasa saturada en percentiles altos dentro de su categoría, fi..."
1,7501030473022,True,"TOSTIS CON FIBRA: azúcar/sal/grasa saturada en percentiles bajos, fibra/proteína altas, NOVA 1 s..."
2,0000654193184,True,Milton's Galletas Saladas Multigrano: allergens='en:gluten' confirmado. Alérgica a gluten debe s...
3,0010248765159,True,"Fideo mediano: gluten SOLO en traces ('puede contener'), no en allergens. El fail-safe (A15) deb..."
4,00000140,True,tisane saveur lunaire: allergens y traces ambos vacíos. Sin ningún dato de alérgenos debe ser no...
5,0000880688789,True,Mister Natural - Datil Premium: ingredients_analysis_tags trae 'en:vegan' confirmado. Dieta vega...
6,0000182602460,True,Isabar: ingredients_analysis_tags trae 'en:non-vegan' confirmado. Dieta vegana debe salir incomp...
7,0010248765135,True,Yemina Semilla de melón: ingredients_analysis_tags trae 'en:maybe-vegan' ('tal vez'). Dieta vega...
8,00000154,True,"Mct Oil Powder: sin categoria_referencia resoluble, pero SÍ tiene nova_group. D1 debe ser None; ..."
9,0000880688787,True,Mister Natural - Chabacano Deshidratado: tiene categoria_referencia (en:dried-fruits) y dato nut...



Todos los casos pasan.


## Sección 2 — Parity-check con Nutri-Score (decisión A8/A30)

D1 y Nutri-Score puntúan nutrientes parcialmente distintos y con metodologías distintas (ver
docstring de `evaluacion/parity_check.py`): no se espera una correlación fuerte, solo que el orden
vaya "a grandes rasgos" en la misma dirección. Se usa Spearman (vía rangos + Pearson, sin añadir
`scipy` como dependencia) entre D1 (más alto = mejor) y `nutriscore_score` (más alto = peor, así
que se espera una correlación **negativa**), más el promedio de D1 por `nutriscore_grade`
('a' a 'e') para verificar que decrece de forma monótona (sin subir en ningún paso).

In [3]:
print("Distribución de nutriscore_grade en el universo México:")
print(df["nutriscore_grade"].value_counts(dropna=False).to_dict())
print(f"\nUmbral de correlación razonable (A30): correlacion_spearman <= {UMBRAL_CORRELACION_RAZONABLE}\n")

reporte_parity = calcular_parity_check(df["d1"], df["nutriscore_score"], df["nutriscore_grade"])

print(f"n (productos con D1 y nutriscore_score) = {reporte_parity['n']}")
print(f"correlación de Spearman = {reporte_parity['correlacion_spearman']:.4f}")
print(f"¿correlación razonable? = {reporte_parity['correlacion_razonable']}")
print(f"¿monótono por grado? = {reporte_parity['monotono_por_grado']}")
print("\nD1 promedio por grado Nutri-Score:")
display(reporte_parity["promedio_d1_por_grado"].round(3))

assert reporte_parity["correlacion_razonable"], "correlación fuera del umbral esperado: revisar D1"
assert reporte_parity["monotono_por_grado"], "el promedio de D1 no decrece de forma monótona por grado"

Distribución de nutriscore_grade en el universo México:
{'unknown': 9845, 'e': 1759, 'd': 1489, 'c': 1409, 'a': 940, 'b': 834, 'not-applicable': 444, nan: 131}

Umbral de correlación razonable (A30): correlacion_spearman <= -0.3

n (productos con D1 y nutriscore_score) = 6041
correlación de Spearman = -0.4747
¿correlación razonable? = True
¿monótono por grado? = True

D1 promedio por grado Nutri-Score:


nutriscore_grade
a    59.617
b    54.811
c    50.844
d    49.001
e    41.570
Name: d1, dtype: float64

**Resultado**: sobre 6.041 productos con D1 y `nutriscore_score` simultáneamente disponibles, la
correlación de Spearman es ≈ −0,4747 — negativa como se esperaba, y por encima en magnitud del
umbral laxo de A30 (−0,3), a pesar de que D1 solo cubre 5 de los nutrientes que pesa Nutri-Score.
El promedio de D1 decrece de forma estrictamente monótona de grado 'a' (≈59,6) a 'e' (≈41,6), sin
ningún salto hacia arriba. Ambas señales apuntan en la misma dirección: D1 no contradice a
Nutri-Score, lo cual es la verificación de sanidad que A8 pide — no una validación de que D1 deba
replicar a Nutri-Score, que sería incorrecto dado A4 (Nutri-Score no puntúa en NutriMatch
justamente para no contar dos veces los mismos nutrientes).

## Sección 3 — Diagnóstico de cobertura (decisión A8/A31)

Se reutilizan los mismos 3 perfiles de `04_modelo_recomendacion.ipynb` (Ana, Beto, Caro; ver esa
sección 1 para el detalle completo) para calcular, por perfil, cuántos productos caen en la banda
"información insuficiente" (`cov < 0.5`, A2) y por qué combinación de dimensiones faltantes —
sobre el universo completo de 16.851 productos, igual que la Sección 6 de ese notebook.

In [4]:
PERFILES = {
    "Ana": {
        "alergenos_declarados": ["en:gluten"],
        "dieta_declarada": "vegano",
        "etiquetas_valoradas": ["en:organic", "en:no-gluten"],
        "orden_prioridades": ["D1", "D3", "D2"],
    },
    "Beto": {
        "alergenos_declarados": [],
        "dieta_declarada": None,
        "etiquetas_valoradas": ["en:fair-trade"],
        "orden_prioridades": ["D2", "D1", "D3"],
    },
    "Caro": {
        "alergenos_declarados": ["en:milk", "en:soybeans"],
        "dieta_declarada": "vegetariano",
        "etiquetas_valoradas": [],
        "orden_prioridades": ["D3", "D1", "D2"],
    },
}

for nombre, perfil in PERFILES.items():
    perfil["pesos"] = convertir_prioridades_a_pesos(perfil["orden_prioridades"])
    df[f"d3_{nombre}"] = df["labels_tags"].apply(lambda lt, etq=perfil["etiquetas_valoradas"]: calcular_d3(lt, etq))

pd.DataFrame({nombre: perfil["pesos"] for nombre, perfil in PERFILES.items()}).T.round(4)

,D1,D3,D2
Ana,0.5000,0.3333,0.1667
Beto,0.3333,0.1667,0.5000
Caro,0.3333,0.5000,0.1667


In [5]:
resumen_diagnostico = {}
desgloses = {}

for nombre, perfil in PERFILES.items():
    disponibilidad = pd.DataFrame(
        {
            "D1": df["d1"].notna(),
            "D2": df["d2"].notna(),
            "D3": df[f"d3_{nombre}"].notna(),
        }
    )
    reporte = diagnosticar_cobertura(disponibilidad, perfil["pesos"])
    desgloses[nombre] = reporte["desglose_por_combinacion_faltante"]
    resumen_diagnostico[nombre] = {
        "n_total": reporte["n_total"],
        "n_informacion_insuficiente": reporte["n_informacion_insuficiente"],
        "pct_informacion_insuficiente": reporte["porcentaje_informacion_insuficiente"],
    }

pd.DataFrame(resumen_diagnostico).T.round(2)

,n_total,n_informacion_insuficiente,pct_informacion_insuficiente
Ana,16851.0,9517.0,56.48
Beto,16851.0,9632.0,57.16
Caro,16851.0,10974.0,65.12


In [6]:
for nombre, desglose in desgloses.items():
    print(f"--- {nombre} — desglose por combinación de dimensiones faltantes ---")
    display(desglose.round(2))
    print()

--- Ana — desglose por combinación de dimensiones faltantes ---


,dimensiones_faltantes,n,porcentaje_del_total,porcentaje_de_insuficientes
0,D1+D2+D3,7835,46.50,82.33
1,D1+D2,1072,6.36,11.26
2,D1+D3,610,3.62,6.41



--- Beto — desglose por combinación de dimensiones faltantes ---


,dimensiones_faltantes,n,porcentaje_del_total,porcentaje_de_insuficientes
0,D1+D2+D3,7835,46.50,81.34
1,D1+D2,1072,6.36,11.13
2,D2+D3,725,4.30,7.53



--- Caro — desglose por combinación de dimensiones faltantes ---


,dimensiones_faltantes,n,porcentaje_del_total,porcentaje_de_insuficientes
0,D1+D2+D3,8907,52.86,81.16
1,D2+D3,1163,6.90,10.60
2,D1+D3,904,5.36,8.24


**Resultado**: la banda "información insuficiente" va de 56,5 % (Ana) a 65,1 % (Caro) del universo
de 16.851 productos — cifras idénticas a las de `04_modelo_recomendacion.ipynb` (Sección 6), como
debía ser: es el mismo cálculo, solo que aquí se desglosa el motivo. Para los tres perfiles, la
combinación dominante entre los insuficientes es **"D1+D2+D3" (faltan las tres)**: 81-82 % de los
insuficientes de Ana y Beto, y 81,2 % de los de Caro. Es decir, la causa principal de
"información insuficiente" no es una dimensión aislada débil, sino productos que carecen por
completo de dato nutricional Y de NOVA a la vez (D1 y D2 comparten esa cobertura baja del 40-42 %,
Sección de setup) — y para quien no valora ninguna etiqueta (Caro) eso además arrastra a D3.
La segunda causa más común difiere por perfil: para Ana y Beto es "D1+D2" (11,3 %, productos con
D3 disponible pero sin nutrición ni NOVA); para Caro es "D2+D3" (10,6 %, coherente con que D3 es
`None` para el 100 % de sus productos — Sección 5 de `04_modelo_recomendacion.ipynb` — así que
cualquier producto sin D2 tampoco tiene D3, por definición, para ella). Ninguna combinación de una
sola dimensión faltante aparece en la tabla: con solo 2 de 3 pesos disponibles ya alcanza el
umbral de 0,5 en casi todos los repartos de A25 (0,50+0,33=0,83; 0,50+0,17=0,67; 0,33+0,17=0,5
exacto), así que perder una sola dimensión rara vez basta para caer en la banda.

## Sección 4 — Síntesis: decisiones confirmadas en el paso 8

Verificado contra `off_csv_20260919` (16.851 productos). Estas decisiones se registran en
`AGENTS.md` como A29-A31 y B14, siguiendo el mismo patrón que A21-A27 de los pasos 6 y 7.

### A29. Golden set: 13 casos reales, revisados a mano, sobre productos del snapshot

Confirmado: 13/13 casos pasan (Sección 1). Cubren D1/D2 en ambos extremos, los tres estados de
alergia (confirmado en `allergens`, "puede contener" en `traces`, sin dato), los tres estados de
dieta vegana (compatible/incompatible/"tal vez"→no verificable), independencia entre D1 y D2
cuando falta solo una fuente, el fallback de nombre (A28), D3 con una etiqueta exacta, y un caso
de regresión de score completo. Implementado en `evaluacion/golden_set.py`, con pruebas de
regresión en `tests/test_golden_set.py` que leen el Parquet real versionado (no hace falta
descargar nada para correrlas). Si el snapshot se regenera y un caso deja de cumplirse, es señal
de revisar ese caso puntual (¿sigue siendo válido con el dato nuevo?), no necesariamente un bug.

### A30. Parity-check: correlación de Spearman D1 vs `nutriscore_score`, umbral −0,3

Confirmado sobre 6.041 productos con ambos datos: correlación de Spearman ≈ −0,4747 (negativa,
como se esperaba, y más allá en magnitud del umbral laxo de −0,3), y el promedio de D1 decrece de
forma estrictamente monótona de grado 'a' (≈59,6) a 'e' (≈41,6). El umbral es deliberadamente laxo
porque D1 (5 nutrientes) y Nutri-Score (más nutrientes, distinta metodología) no deberían
correlacionar fuerte — el chequeo es de sanidad ("misma dirección"), no de réplica. Spearman se
calcula sobre rangos + Pearson (`Series.rank().corr()`) para no añadir `scipy` como dependencia
nueva solo por esta función. Implementado en `evaluacion/parity_check.py`.

### A31. Diagnóstico de cobertura: desglose por combinación de dimensiones faltantes

Confirmado: para los 3 perfiles de ejemplo, "información insuficiente" va de 56,5 % a 65,1 % del
universo (idéntico a `04_modelo_recomendacion.ipynb`), y la causa dominante en los tres es la
combinación "D1+D2+D3" (81-82 % de los insuficientes): productos sin dato nutricional NI NOVA a la
vez. Ninguna combinación de una sola dimensión faltante aparece porque, con la fórmula de pesos de
A25 (0,50/0,33/0,17), cualquier par de dimensiones ya suma ≥ 0,5 exacto — perder una sola nunca
basta para cruzar el umbral. Implementado en `evaluacion/coverage_diagnostic.py`.

### B14. `nutriscore_score` llega como VARCHAR en el snapshot crudo, no como número

Verificado al construir este notebook (Sección de setup): `scripts/ingesta_off.py` usa
`all_varchar=true` a propósito en la ingesta ("la conversión de tipos se hace después, de forma
auditada"). `nutriscore_score` no es parte de CORE8, así que el paso 6 (`sanitize.py`) nunca lo
tipó — sigue siendo texto en `off_mexico_20260919.parquet`. Sin una conversión explícita con
`pd.to_numeric`, cualquier comparación u ordenamiento de esta columna se haría por orden
LEXICOGRÁFICO (`"9" > "51"`) en vez de numérico, en silencio: la primera versión de este notebook
calculó una correlación de Spearman con el valor sin convertir y obtuvo −0,2727 en vez de −0,4747
(una diferencia de casi el doble, suficiente para haber quedado por debajo del umbral de A30 y dar
un falso negativo). Cualquier módulo futuro que use `nutriscore_score` (o cualquier otra columna
del snapshot crudo que no pase por `sanitize.py`) debe convertirla explícitamente antes de
compararla numéricamente.